In [51]:
!jupyter nbconvert --to notebook --execute preprocess.ipynb --inplace

[NbConvertApp] Converting notebook preprocess.ipynb to notebook
[NbConvertApp] Writing 121494 bytes to preprocess.ipynb


In [200]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pulp import *
from scipy.stats import poisson, norm
from pulp import LpProblem, LpMaximize, LpVariable, LpStatus, lpSum, value, PULP_CBC_CMD

In [201]:
OUT_CSV = "fantasy_enriched.csv"



| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |



## Core identity / player info

| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | Unique FIFA fantasy player ID |
| `name` | str | Player name |
| `squad_id` | int | Internal squad/nation ID |
| `team` | str | National team |
| `position` | str | DEF / MID / FWD / GK |
| `price` | float | Fantasy price |
| `status` | str | Availability status |
| `total_points` | int | Total tournament points |
| `avg_points` | float | Average points per match |
| `matches_played` | int | Matches with participation |
| `percent_selected` | float | Selection rate (%) |
| `next_fixture` | int | Next match ID |


## Round fantasy points

| Column | Type | Description |
|---|---|---|
| `round_1_points` | int | Points in round 1 |
| `round_2_points` | int | Points in round 2 |
| `round_3_points` | int | Points in round 3 |
| `round_4_points` | int | Points in round 4 |


## Betting / scorer mapping

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Matched Betano market name |
| `betano_match_score` | float | Name matching confidence (0–100) |
| `anytime_scorer_prob` | float | Probability player scores anytime |
| `first_scorer_prob` | float | Probability scores first goal |
| `last_scorer_prob` | float | Probability scores last goal |


## Match context

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team |
| `match_away` | str | Away team |
| `is_home` | bool | Player team is home |
| `opponent` | str | Opponent team |


## Match probabilities 

| Column | Type | Description |
|---|---|---|
| `match_home_win_prob` | float | Home win probability |
| `match_draw_prob` | float | Draw probability |
| `match_away_win_prob` | float | Away win probability |
| `match_btts_prob` | float | Both teams score probability |
| `match_over_25_prob` | float | Over 2.5 goals probability |
| `match_over_35_prob` | float | Over 3.5 goals probability |
| `team_score_prob` | float | Team scores ≥1 goal probability |
| `team_score_2_prob` | float | Team scores ≥2 goals probability |
| `team_cs_prob` | float | Team clean sheet probability |
| `opp_score_prob` | float | Opponent scores ≥1 goal probability |
| `opp_cs_prob` | float | Opponent clean sheet probability |
| `team_over_05_prob` | float | Team scores ≥1 goal probability |
| `team_over_15_prob` | float | Team scores ≥2 goals probability |
| `team_qualify_prob` | float | Team advances probability |
| `team_win2_prob` | float | Team wins by 2+ goals probability |
| `opp_over_05_prob` | float | Opponent scores ≥1 goal probability |


## Raw cumulative stats

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals |
| `stat_AS` | int | Assists |
| `stat_CS` | int | Clean sheets |
| `stat_GC` | int | Goals conceded |
| `stat_MP` | int | Minutes played |
| `stat_YC` | int | Yellow cards |
| `stat_RC` | int | Red cards |
| `stat_ST` | int | Shots on target |
| `stat_SB` | int | Shots blocked |
| `stat_CC` | int | Chances created |
| `stat_PS` | int | Penalties saved |
| `stat_T` | int | Tackles |
| `stat_S` | int | Saves |
| `stat_SXI` | int | Starts (XI appearances) |
| `stat_OG` | int | Own goals |
| `stat_PC` | int | Penalties committed |
| `stat_PW` | int | Penalties won |
| `stat_FK` | int | Free kicks won |



## Round-by-round stats (1–4)

| Pattern | Type | Description |
|---|---|---|
| `round_i_SXI` | int | Started (0/1) |
| `round_i_MP` | int | Minutes |
| `round_i_AS` | int | Assists |
| `round_i_YC` | int | Yellow cards |
| `round_i_RC` | int | Red cards |
| `round_i_OG` | int | Own goals |
| `round_i_PW` | int | Penalties won |
| `round_i_PC` | int | Penalties committed |
| `round_i_CS` | int | Clean sheets |
| `round_i_GS` | int | Goals |
| `round_i_GC` | int | Goals conceded |
| `round_i_PS` | int | Penalties saved |
| `round_i_T` | int | Tackles |
| `round_i_CC` | int | Chances created |
| `round_i_ST` | int | Shots on target |
| `round_i_FK` | int | Free kicks |
| `round_i_S` | int | Saves |
| `round_i_SB` | int | Shots blocked |



## External matching / enrichment

| Column | Type | Description |
|---|---|---|
| `clubstats_matched_player` | str | External player match |
| `clubs_match_dist` | float | Matching distance score |



## Form / usage trends

| Column | Type | Description |
|---|---|---|
| `started_last_match` | int | Started last match |
| `starts_last_3` | int | Starts last 3 matches |
| `starts_last_5` | int | Starts last 5 matches |
| `start_rate` | float | Start frequency |
| `minutes_last_3_avg` | float | Avg minutes last 3 |
| `minutes_last_5_avg` | float | Avg minutes last 5 |



## Performance rates

| Column | Type | Description |
|---|---|---|
| `goal_rate` | float | Goals per minute |
| `goals_per_start` | float | Goals per start |
| `shots_on_target_per90` | float | SOT per 90 |
| `goals_per_shot_on_target` | float | Conversion rate |
| `assist_rate` | float | Assists per minute |
| `chance_created_per90` | float | Chances per 90 |
| `tackles_per90` | float | Tackles per 90 |
| `cc_per90` | float | Chances created per 90 |
| `sot_per90` | float | Shots on target per 90 |
| `saves_per90` | float | Saves per 90 |
| `yc_per90` | float | Yellow cards per 90 |
| `rc_per90` | float | Red cards per 90 |
| `penalty_conceded_rate` | float | Penalties conceded per 90 |
| `own_goal_rate` | float | Own goals per 90 |
| `penalties_won_per90` | float | Penalties won per 90 |



## Recent / streak features

| Column | Type | Description |
|---|---|---|
| `recent_goals_last3` | int | Goals last 3 matches |
| `goal_streak` | int | Consecutive scoring matches |
| `recent_assists_last3` | int | Assists last 3 matches |
| `recent_chances_created_per90` | float | Recent creation rate |
| `recent_cs_rate` | float | Clean sheet rate |
| `recent_tackles_per90` | float | Recent tackles |
| `recent_cc_per90` | float | Recent chances created |
| `recent_sot_per90` | float | Recent shots on target |
| `recent_saves_per90` | float | Recent saves |
| `recent_penalties_won` | int | Penalties won last 3 |



## Expected value features

| Column | Type | Description |
|---|---|---|
| `goal_expectation` | float | Scoring proxy |
| `goal_expectation2` | float | Scoring proxy (2+ goals) |
| `assist_expectation` | float | Assist proxy |
| `cs_expectation` | float | Clean sheet proxy |
| `expected_minutes` | float | Expected minutes |
| `expected_gc_penalty` | float | Defensive penalty |
| `expected_tackle_points` | float | Tackles contribution |
| `expected_cc_points` | float | Chance creation points |
| `expected_sot_points` | float | Shot contribution points |
| `expected_save_points` | float | Save contribution points |



## Fantasy points dynamics

| Column | Type | Description |
|---|---|---|
| `points_last1` | int | Last match points |
| `points_last3_avg` | float | Avg last 3 |
| `points_last5_avg` | float | Avg last 5 |
| `weighted_points` | float | Recency weighted |
| `rolling_std_points` | float | Variance last 5 |
| `rolling_max_points` | int | Max last 5 |



## Team-position ranking features

| Column | Type | Description |
|---|---|---|
| `price_rank_team_position` | float | Price rank |
| `selected_rank_team_position` | float | Selection rank |
| `points_rank_team_position` | float | Points rank |
| `minutes_rank_team_position` | float | Minutes rank |
| `starts_rank_team_position` | float | Starts rank |
| `anytime_rank_team_position` | float | Scoring odds rank |
| `shots_rank_team_position` | float | Shooting rank |
| `cc_rank_team_position` | float | Chance creation rank |
| `tackles_rank_team_position` | float | Tackles rank |

In [202]:
df= pd.read_csv("fantasy_enriched.csv")

In [203]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [204]:
df = df[df["status"] == "playing"].copy()

In [205]:
df.loc[df["name"] == "Johan Manzambi", "status"] = "injured"  # :(
df.loc[df["name"] == "Nicolás Tagliafico", "status"] = "injured"  # :(


In [206]:
# probability of playing a minute

raw_any = (
    0.2* df["minutes_rank_team_position"]
  + 0.10 * df["stat_MP"]        # minutes needs to be very team stratified since while for other point sources players are all eligible
  + 0.15 * df["starts_rank_team_position"]
  + 0.10 * df["price_rank_team_position"]     # for minutes its a team stratified signal.
  + 0.1 * df["selected_rank_team_position"]
  + 0.25 * df["minutes_last_3_avg"]
  + 0.1 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_any"] = sigmoid((raw_any - 0.5) * 6) 

In [207]:
df.loc[
    :,
    [
        "prob_plays_any",
        "name",
        "team",
        "position",
        "stat_MP",
        "minutes_rank_team_position",
        "starts_rank_team_position",
        "selected_rank_team_position",
        "price_rank_team_position",
        "anytime_rank_team_position",
    ],
].sort_values("prob_plays_any", ascending=False).head(30)

,prob_plays_any,name,team,position,stat_MP,minutes_rank_team_position,starts_rank_team_position,selected_rank_team_position,price_rank_team_position,anytime_rank_team_position
9,0.937027,Emiliano Martínez,Argentina,GK,1.000000,1.000000,0.666667,1.000000,1.000000,1.000000
4,0.930377,Lionel Messi,Argentina,FWD,0.883333,1.000000,0.625000,1.000000,1.000000,1.000000
26,0.923055,Harry Kane,England,FWD,0.938333,1.000000,0.625000,1.000000,1.000000,1.000000
18,0.922397,Alexis Mac Allister,Argentina,MID,0.898333,1.000000,0.545455,1.000000,0.909091,0.136364
80,0.914217,Lisandro Martínez,Argentina,DEF,0.850000,1.000000,0.562500,1.000000,0.750000,0.250000
46,0.905805,Kylian Mbappé,France,FWD,0.863333,1.000000,0.666667,1.000000,1.000000,1.000000
70,0.900628,Mikel Oyarzabal,Spain,FWD,0.781667,1.000000,0.625000,1.000000,1.000000,1.000000
19,0.897104,Enzo Fernández,Argentina,MID,0.800000,0.909091,0.545455,0.900000,1.000000,0.136364
38,0.895923,Jude Bellingham,England,MID,0.858333,0.900000,0.550000,1.000000,0.900000,1.000000
59,0.893681,Michael Olise,France,MID,0.813333,1.000000,0.545455,1.000000,0.909091,0.772727


In [208]:
# probability of starting / playing 60 minutes

raw_any = (
    0.25 * df["minutes_rank_team_position"]
  + 0.05 * df["starts_rank_team_position"]
  + 0.20 * df["stat_MP"]
  + 0.05 * df["selected_rank_team_position"]
  + 0.40 * df["minutes_last_3_avg"]
  + 0.05 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_60min"] = sigmoid((raw_any - 0.5) * 6)


In [209]:
cols = [
    "prob_plays_60min",
    "name",
    "team",
    "position",
    "minutes_rank_team_position",
    "starts_rank_team_position",
    "price_rank_team_position",
    "selected_rank_team_position",
    "minutes_last_3_avg",
    "round_5_MP",
    "anytime_rank_team_position",
]

df[cols].sort_values("prob_plays_60min", ascending=False).head(30)

,prob_plays_60min,name,team,position,minutes_rank_team_position,starts_rank_team_position,price_rank_team_position,selected_rank_team_position,minutes_last_3_avg,round_5_MP,anytime_rank_team_position
9,0.947846,Emiliano Martínez,Argentina,GK,1.000000,0.666667,1.000000,1.000000,1.000000,1.000000,1.000000
4,0.939772,Lionel Messi,Argentina,FWD,1.000000,0.625000,1.000000,1.000000,1.000000,1.000000,1.000000
18,0.939439,Alexis Mac Allister,Argentina,MID,1.000000,0.545455,0.909091,1.000000,1.000000,1.000000,0.136364
80,0.936360,Lisandro Martínez,Argentina,DEF,1.000000,0.562500,0.750000,1.000000,1.000000,1.000000,0.250000
26,0.930088,Harry Kane,England,FWD,1.000000,0.625000,1.000000,1.000000,0.906061,0.988889,1.000000
32,0.910668,Jordan Pickford,England,GK,1.000000,0.666667,1.000000,1.000000,0.909091,1.000000,0.666667
38,0.905626,Jude Bellingham,England,MID,0.900000,0.550000,0.900000,1.000000,0.881818,1.000000,1.000000
41,0.904273,Dayot Upamecano,France,DEF,1.000000,0.555556,0.833333,0.877778,0.818182,1.000000,0.888889
19,0.903705,Enzo Fernández,Argentina,MID,0.909091,0.545455,1.000000,0.900000,0.909091,1.000000,0.136364
36,0.903674,Elliot Anderson,England,MID,1.000000,0.550000,0.300000,0.450000,0.863636,0.833333,0.200000


There is a phenomenon where a player will be overvalued, such as Lukaku here for belgium. He has not played much, but since Belgium only has 3 forward and only him has significative minutes he ranks number 1 for all ranks in BEL,FWD. Need to include more general sums and beware with manual selection.

In [210]:
df["lambda_goal"] = -np.log(
    (1 - df["anytime_scorer_prob"]).clip(lower=0.001)
)

lambda_adj = (
    0.70 * df["lambda_goal"]
  + 0.1 * df["recent_goals_last3"]
  + 0.02 * df["team_score_2_prob"]
  + 0.1 * df["stat_GS"]
  + 0.05 * df["stat_ST"]
  + 0.03 * df["price"]
)

df["prob_scores"] = (
    lambda_adj.clip(lower=0)
    * df["prob_plays_60min"]
    * 1.5
)

df.loc[df["position"] == "GK", "prob_scores"] = 0.
df.loc[df["position"] == "DEF", "prob_scores"] = df["prob_scores"] * 0.7

In [211]:
cols = [
    "name",
    "team",
    "position",
    "prob_scores",
    "goal_expectation",
    "anytime_scorer_prob",
    "recent_goals_last3",
    "recent_sot_per90",
    "goal_rate",
    "shots_on_target_per90",
    "stat_GS",
    "stat_ST",
    "price_rank_team_position",
]

df[cols].sort_values("prob_scores", ascending=False).head(30)

,name,team,position,prob_scores,goal_expectation,anytime_scorer_prob,recent_goals_last3,recent_sot_per90,goal_rate,shots_on_target_per90,stat_GS,stat_ST,price_rank_team_position
46,Kylian Mbappé,France,FWD,1.053034,0.369430,0.5051,1.00,0.555556,0.926641,0.770270,1.000,1.000000,1.000000
4,Lionel Messi,Argentina,FWD,0.826436,0.255103,0.3968,0.50,0.381818,0.905660,0.713208,1.000,0.947368,1.000000
26,Harry Kane,England,FWD,0.792733,0.268069,0.3922,0.75,0.280936,0.639432,0.447602,0.750,0.631579,1.000000
70,Mikel Oyarzabal,Spain,FWD,0.648396,0.259953,0.3774,0.50,0.325581,0.511727,0.447761,0.500,0.526316,1.000000
38,Jude Bellingham,England,MID,0.576881,0.164723,0.2410,1.00,0.384880,0.699029,0.448544,0.750,0.578947,0.900000
47,Ousmane Dembélé,France,MID,0.524292,0.228563,0.3125,0.25,0.056225,0.649351,0.272727,0.625,0.315789,1.000000
68,Lamine Yamal,Spain,MID,0.425942,0.211944,0.3077,0.00,0.422642,0.148148,0.518519,0.125,0.526316,1.000000
5,Julián Alvarez,Argentina,FWD,0.345912,0.160725,0.2500,0.25,0.105263,0.146699,0.205379,0.125,0.210526,0.500000
59,Michael Olise,France,MID,0.327076,0.176267,0.2410,0.00,0.158491,0.000000,0.215164,0.000,0.263158,0.909091
87,Bradley Barcola,France,MID,0.240092,0.176267,0.2410,0.25,0.256098,0.425532,0.297872,0.250,0.210526,0.772727


In [212]:
prob_assists = (
    0.5 * df["chance_created_per90"]
  + 0.15 * df["team_score_2_prob"]
  + 0.1 * df["prob_scores"]
  + 0.1 * df["assist_rate"]
  + 0.15 * df["stat_AS"]
)

df["lambda_assist"] = -np.log(
    (1 - prob_assists).clip(lower=0.001)
)

df["expected_assists"] = df["lambda_assist"].clip(lower=0) * df["prob_plays_60min"] 

In [213]:
df.loc[df["position"] == "GK", "expected_assists"] = df["expected_assists"] * 0.1
df.loc[df["position"] == "DEF", "expected_assists"] = df["expected_assists"] * 0.9

In [214]:
cols = [
    "name",
    "team",
    "position",
    "expected_assists",
    "assist_expectation",
    "assist_rate",
    "chance_created_per90",
    "team_score_2_prob",
    "recent_assists_last3",
    "prob_plays_60min",
    "stat_AS",
]

df[cols].sort_values("expected_assists", ascending=False).head(30)

,name,team,position,expected_assists,assist_expectation,assist_rate,chance_created_per90,team_score_2_prob,recent_assists_last3,prob_plays_60min,stat_AS
4,Lionel Messi,Argentina,FWD,0.878668,0.764202,0.067925,0.805660,0.3676,0.666667,0.939772,0.4
59,Michael Olise,France,MID,0.795393,0.674447,0.184426,0.625000,0.4950,0.666667,0.897238,1.0
47,Ousmane Dembélé,France,MID,0.429667,0.427442,0.077922,0.396104,0.4950,0.333333,0.862021,0.4
27,Bukayo Saka,England,MID,0.427745,0.691180,0.202247,0.685393,0.4049,0.333333,0.567316,0.6
46,Kylian Mbappé,France,FWD,0.371335,0.127077,0.104247,0.117761,0.4950,0.333333,0.897406,0.6
38,Jude Bellingham,England,MID,0.362126,0.358340,0.034951,0.355340,0.4049,0.000000,0.905626,0.2
66,Marc Cucurella,Spain,DEF,0.337336,0.344400,0.100000,0.338889,0.4082,0.666667,0.901911,0.6
37,Declan Rice,England,MID,0.332994,0.478096,0.046632,0.474093,0.4049,0.000000,0.785850,0.2
29,Anthony Gordon,England,MID,0.321964,0.376239,0.165138,0.373089,0.4049,1.000000,0.693773,0.6
78,Dani Olmo,Spain,MID,0.312155,0.362526,0.105263,0.356725,0.4082,0.333333,0.770041,0.4


In [215]:
position_yc_modifier = {
    "GK": 0.15,
    "FWD": 0.35,
    "DEF": 0.45,
    "MID": 0.55,
}

df["position_yc_modifier"] = df["position"].map(position_yc_modifier)

raw_yc = (
    0.4 * df["yc_per90"]
  + 0.25 * df["tackles_per90"]
  + 0.20 * df["opp_over_05_prob"]
  + 0.05 * df["rc_per90"]
  + 0.1 * df["position_yc_modifier"]
)

df["prob_yellow_card"] = raw_yc * df["prob_plays_60min"]

In [216]:
cols = [
    "name",
    "team",
    "position",
    "prob_yellow_card",
    "yc_per90",
    "stat_YC",
    "tackles_per90",
    "recent_tackles_per90",
    "opp_over_05_prob",
    "prob_plays_60min",
    "rc_per90",
]

df[cols].sort_values("prob_yellow_card", ascending=False).head(30)

,name,team,position,prob_yellow_card,yc_per90,stat_YC,tackles_per90,recent_tackles_per90,opp_over_05_prob,prob_plays_60min,rc_per90
77,Rodri,Spain,MID,0.276396,0.000000,0.0,0.212291,0.333333,1.000000,0.897180,0.0
79,Pedri,Spain,MID,0.245412,0.041958,0.5,0.125874,0.172249,1.000000,0.809270,0.0
68,Lamine Yamal,Spain,MID,0.241702,0.000000,0.0,0.118519,0.203774,1.000000,0.849180,0.0
67,Pau Cubarsí,Spain,DEF,0.238498,0.033333,0.5,0.033333,0.100000,1.000000,0.894366,0.0
64,Aymeric Laporte,Spain,DEF,0.236441,0.033395,0.5,0.055659,0.133829,1.000000,0.868398,0.0
66,Marc Cucurella,Spain,DEF,0.228484,0.000000,0.0,0.033333,0.033333,1.000000,0.901911,0.0
70,Mikel Oyarzabal,Spain,FWD,0.223655,0.000000,0.0,0.063966,0.104651,1.000000,0.891084,0.0
63,Pedro Porro,Spain,DEF,0.223623,0.000000,0.0,0.166667,0.233333,1.000000,0.780081,0.0
78,Dani Olmo,Spain,MID,0.216625,0.000000,0.0,0.105263,0.185950,1.000000,0.770041,0.0
61,Álex Baena,Spain,MID,0.213573,0.054878,0.5,0.128049,0.223881,1.000000,0.691256,0.0


In [217]:
raw_pw = (
    0.10 * df["stat_PW"]                    
  + 0.3 * df["anytime_scorer_prob"]      
  + 0.10 * df["chance_created_per90"]     
  + 0.2 * df["stat_CC"]                  # recent form in 
  + 0.2 * df["team_score_2_prob"]        # team expected to attack heavily
  + 0.10 * df["price"] # quality proxy
)

df["prob_pen_won"] = 0.4 * sigmoid((raw_pw - 0.65) * 5) * df["prob_plays_60min"]

In [218]:
cols = [
    "name",
    "team",
    "position",
    "prob_pen_won",
    "stat_PW",
    "anytime_scorer_prob",
    "assist_expectation",
    "chance_created_per90",
    "team_score_2_prob",
    "price_rank_team_position",
    "prob_plays_60min",
]

df[cols].sort_values("prob_pen_won", ascending=False).head(30)

,name,team,position,prob_pen_won,stat_PW,anytime_scorer_prob,assist_expectation,chance_created_per90,team_score_2_prob,price_rank_team_position,prob_plays_60min
4,Lionel Messi,Argentina,FWD,0.148800,0.0,0.3968,0.764202,0.805660,0.3676,1.000000,0.939772
46,Kylian Mbappé,France,FWD,0.111625,1.0,0.5051,0.127077,0.117761,0.4950,1.000000,0.897406
59,Michael Olise,France,MID,0.102436,0.0,0.2410,0.674447,0.625000,0.4950,0.909091,0.897238
47,Ousmane Dembélé,France,MID,0.082318,0.0,0.3125,0.427442,0.396104,0.4950,1.000000,0.862021
26,Harry Kane,England,FWD,0.064615,0.0,0.3922,0.109263,0.108348,0.4049,1.000000,0.930088
38,Jude Bellingham,England,MID,0.063382,0.0,0.2410,0.358340,0.355340,0.4049,0.900000,0.905626
29,Anthony Gordon,England,MID,0.054620,1.0,0.1818,0.376239,0.373089,0.4049,0.450000,0.693773
83,Lautaro Martínez,Argentina,FWD,0.053993,1.0,0.2564,0.574778,0.605960,0.3676,0.750000,0.465996
66,Marc Cucurella,Spain,DEF,0.053312,0.0,0.0833,0.344400,0.338889,0.4082,0.625000,0.901911
70,Mikel Oyarzabal,Spain,FWD,0.052660,0.0,0.3774,0.132179,0.130064,0.4082,1.000000,0.891084


In [219]:
raw_cs = (
    0.80 * df["team_cs_prob"]            # strongest signal
  + 0.05 * df["stat_CS"]               # tournament history
  + 0.15 * (1 - df["stat_GC"])         # fewer goals conceded is better
)

base_cs = sigmoid((raw_cs - 0.5) * 6)

df["prob_clean_sheet"] = base_cs * df["prob_plays_60min"]

df.loc[df["position"] == "FWD", "prob_clean_sheet"] = 0.0

In [220]:
cols = [
    "name",
    "team",
    "position",
    "prob_clean_sheet",
    "team_cs_prob",
    "prob_plays_60min",
    "stat_CS",
    "stat_GC",
    "minutes_rank_team_position",
    "tackles_per90",
]

df[cols].sort_values("prob_clean_sheet", ascending=False).head(30)

,name,team,position,prob_clean_sheet,team_cs_prob,prob_plays_60min,stat_CS,stat_GC,minutes_rank_team_position,tackles_per90
47,Ousmane Dembélé,France,MID,0.324185,0.3112,0.862021,0.833333,0.166667,0.909091,0.038961
85,William Saliba,France,DEF,0.312590,0.3112,0.857780,0.666667,0.166667,0.777778,0.026667
44,Lucas Digne,France,DEF,0.309316,0.3112,0.773649,0.666667,0.000000,0.666667,0.086207
53,Adrien Rabiot,France,MID,0.307629,0.3112,0.844164,0.666667,0.166667,0.818182,0.040000
79,Pedri,Spain,MID,0.303496,0.2686,0.809270,1.000000,0.000000,0.909091,0.125874
41,Dayot Upamecano,France,DEF,0.298799,0.3112,0.904273,0.666667,0.333333,1.000000,0.159696
66,Marc Cucurella,Spain,DEF,0.297125,0.2686,0.901911,0.833333,0.166667,0.937500,0.033333
59,Michael Olise,France,MID,0.296475,0.3112,0.897238,0.666667,0.333333,1.000000,0.086066
77,Rodri,Spain,MID,0.295567,0.2686,0.897180,0.833333,0.166667,1.000000,0.212291
67,Pau Cubarsí,Spain,DEF,0.294640,0.2686,0.894366,0.833333,0.166667,0.937500,0.033333


In [221]:
position_rc_modifier = {
    "GK": 0.04,
    "FWD": 0.08,
    "DEF": 0.12,
    "MID": 0.14,
}

df["position_rc_modifier"] = df["position"].map(position_rc_modifier)

raw_rc = (
    0.10 * df["position_rc_modifier"]   # strongest prior
  + 0.25 * df["rc_per90"]              # direct history
  + 0.15 * df["tackles_per90"]         # physical involvement
  + 0.25 * df["prob_yellow_card"]      # disciplinary tendency
  + 0.25 * df["opp_over_05_prob"]      # more defending -> more challenges
)

df["prob_red_card"] = 0.1 * sigmoid((raw_rc - 0.5) * 5) * df["prob_plays_60min"]

In [222]:
cols = [
    "name",
    "team",
    "position",
    "prob_red_card",
    "position_rc_modifier",
    "rc_per90",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_red_card", ascending=False).head(30)

,name,team,position,prob_red_card,position_rc_modifier,rc_per90,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
77,Rodri,Spain,MID,0.030263,0.14,0.0,0.212291,0.276396,1.000000,0.854084
68,Lamine Yamal,Spain,MID,0.026527,0.14,0.0,0.118519,0.241702,1.000000,0.867872
67,Pau Cubarsí,Spain,DEF,0.026465,0.12,0.0,0.033333,0.238498,1.000000,0.849862
66,Marc Cucurella,Spain,DEF,0.026453,0.12,0.0,0.033333,0.228484,1.000000,0.877989
70,Mikel Oyarzabal,Spain,FWD,0.026079,0.08,0.0,0.063966,0.223655,1.000000,0.900628
64,Aymeric Laporte,Spain,DEF,0.025953,0.12,0.0,0.055659,0.236441,1.000000,0.859420
79,Pedri,Spain,MID,0.025457,0.14,0.0,0.125874,0.245412,1.000000,0.830091
63,Pedro Porro,Spain,DEF,0.024428,0.12,0.0,0.166667,0.223623,1.000000,0.789352
74,Unai Simón,Spain,GK,0.023948,0.04,0.0,0.000000,0.190340,1.000000,0.841212
78,Dani Olmo,Spain,MID,0.023378,0.14,0.0,0.105263,0.216625,1.000000,0.779370


In [223]:
position_og_modifier = {
    "GK": 0.05,
    "FWD": 0.02,
    "MID": 0.04,
    "DEF": 0.08,
}

df["position_og_modifier"] = df["position"].map(position_og_modifier)

raw_og = (
    0.4 * df["position_og_modifier"]
  + 0.5 * df["opp_over_05_prob"]
  + 0.10 * (1 - df["stat_GC"])
)

df["prob_own_goal"] = 0.03 * sigmoid((raw_og - 0.5) * 5) * df["prob_plays_60min"]

In [224]:
cols = [
    "name",
    "team",
    "position",
    "prob_own_goal",
    "position_og_modifier",
    "own_goal_rate",
    "opp_over_05_prob",
    "tackles_per90",
    "stat_GC",
    "prob_plays_any",
]

df[cols].sort_values("prob_own_goal", ascending=False).head(30)

,name,team,position,prob_own_goal,position_og_modifier,own_goal_rate,opp_over_05_prob,tackles_per90,stat_GC,prob_plays_any
66,Marc Cucurella,Spain,DEF,0.017325,0.08,0.0,1.000000,0.033333,0.166667,0.877989
67,Pau Cubarsí,Spain,DEF,0.017180,0.08,0.0,1.000000,0.033333,0.166667,0.849862
77,Rodri,Spain,MID,0.016733,0.04,0.0,1.000000,0.212291,0.166667,0.854084
64,Aymeric Laporte,Spain,DEF,0.016681,0.08,0.0,1.000000,0.055659,0.166667,0.859420
74,Unai Simón,Spain,GK,0.016636,0.05,0.0,1.000000,0.000000,0.166667,0.841212
70,Mikel Oyarzabal,Spain,FWD,0.016366,0.02,0.0,1.000000,0.063966,0.166667,0.900628
68,Lamine Yamal,Spain,MID,0.015837,0.04,0.0,1.000000,0.118519,0.166667,0.867872
79,Pedri,Spain,MID,0.015564,0.04,0.0,1.000000,0.125874,0.000000,0.830091
63,Pedro Porro,Spain,DEF,0.014985,0.08,0.0,1.000000,0.166667,0.166667,0.789352
78,Dani Olmo,Spain,MID,0.014361,0.04,0.0,1.000000,0.105263,0.166667,0.779370


In [225]:
position_pc_modifier = {
    "GK": 0.08,
    "FWD": 0.01,
    "MID": 0.05,
    "DEF": 0.14,
}

df["position_pc_modifier"] = df["position"].map(position_pc_modifier)

raw_pc = (
    0.20 * df["position_pc_modifier"]
  + 0.15 * df["penalty_conceded_rate"]
  + 0.15 * df["tackles_per90"]
  + 0.35 * df["prob_yellow_card"]
  + 0.15 * df["opp_over_05_prob"]
)

df["prob_penalty_committed"] = 0.15 * sigmoid((raw_pc - 0.5) * 5) * df["prob_plays_60min"]

In [226]:
cols = [
    "name",
    "team",
    "position",
    "prob_penalty_committed",
    "position_pc_modifier",
    "penalty_conceded_rate",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_penalty_committed", ascending=False).head(30)

,name,team,position,prob_penalty_committed,position_pc_modifier,penalty_conceded_rate,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
77,Rodri,Spain,MID,0.034703,0.05,0.0,0.212291,0.276396,1.000000,0.854084
67,Pau Cubarsí,Spain,DEF,0.031833,0.14,0.0,0.033333,0.238498,1.000000,0.849862
66,Marc Cucurella,Spain,DEF,0.031674,0.14,0.0,0.033333,0.228484,1.000000,0.877989
64,Aymeric Laporte,Spain,DEF,0.031219,0.14,0.0,0.055659,0.236441,1.000000,0.859420
68,Lamine Yamal,Spain,MID,0.029754,0.05,0.0,0.118519,0.241702,1.000000,0.867872
63,Pedro Porro,Spain,DEF,0.029362,0.14,0.0,0.166667,0.223623,1.000000,0.789352
79,Pedri,Spain,MID,0.028618,0.05,0.0,0.125874,0.245412,1.000000,0.830091
70,Mikel Oyarzabal,Spain,FWD,0.028612,0.01,0.0,0.063966,0.223655,1.000000,0.900628
74,Unai Simón,Spain,GK,0.027624,0.08,0.0,0.000000,0.190340,1.000000,0.841212
78,Dani Olmo,Spain,MID,0.025884,0.05,0.0,0.105263,0.216625,1.000000,0.779370


In [227]:
df["lambda_opp"] = -np.log(df["team_cs_prob"].clip(lower=0.01))

df["prob_gc_0"] = np.exp(-df["lambda_opp"])
df["prob_gc_1"] = df["lambda_opp"] * np.exp(-df["lambda_opp"])
df["prob_gc_2"] = (df["lambda_opp"] ** 2 / 2) * np.exp(-df["lambda_opp"])
df["prob_gc_3"] = (df["lambda_opp"] ** 3 / 6) * np.exp(-df["lambda_opp"])
df["prob_gc_4plus"] = 1 - (
    df["prob_gc_0"]
    + df["prob_gc_1"]
    + df["prob_gc_2"]
    + df["prob_gc_3"]
)

# RAW expected goals conceded (true intensity)
df["expected_goals_conceded"] = df["lambda_opp"]

# Normalize ONLY as auxiliary feature (separate column)
scaler = MinMaxScaler()
df["expected_goals_conceded_norm"] = scaler.fit_transform(
    df[["expected_goals_conceded"]]
)

# Player exposure to goals conceded (correct)
df["expected_player_goals_conceded"] = (
    df["lambda_opp"] * df["prob_plays_60min"]
)

# GC penalty MUST use raw λ (correct Poisson expectation)
df["expected_gc_penalty"] = (
    -(df["lambda_opp"] - 1 + np.exp(-df["lambda_opp"]))
    * df["prob_plays_60min"]
)

df.loc[df["position"].isin(["MID", "FWD"]), [
    "expected_player_goals_conceded",
    "expected_gc_penalty"
]] = 0.0

In [228]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "team_cs_prob",
    "lambda_opp",
    "prob_gc_0",
    "prob_gc_1",
    "prob_gc_2",
    "prob_gc_3",
    "prob_gc_4plus",
    "expected_player_goals_conceded",
    "expected_gc_penalty",
]

df[df["position"].isin(["GK", "DEF"])][cols] \
    .sort_values("expected_gc_penalty") \
    .head(30)

,name,team,position,prob_plays_60min,team_cs_prob,lambda_opp,prob_gc_0,prob_gc_1,prob_gc_2,prob_gc_3,prob_gc_4plus,expected_player_goals_conceded,expected_gc_penalty
66,Marc Cucurella,Spain,DEF,0.901911,0.2686,1.314532,0.2686,0.353083,0.232070,0.101688,0.044559,1.185590,-0.525933
67,Pau Cubarsí,Spain,DEF,0.894366,0.2686,1.314532,0.2686,0.353083,0.232070,0.101688,0.044559,1.175673,-0.521534
74,Unai Simón,Spain,GK,0.885303,0.2686,1.314532,0.2686,0.353083,0.232070,0.101688,0.044559,1.163759,-0.516249
64,Aymeric Laporte,Spain,DEF,0.868398,0.2686,1.314532,0.2686,0.353083,0.232070,0.101688,0.044559,1.141536,-0.506390
63,Pedro Porro,Spain,DEF,0.780081,0.2686,1.314532,0.2686,0.353083,0.232070,0.101688,0.044559,1.025441,-0.454890
9,Emiliano Martínez,Argentina,GK,0.947846,0.3165,1.150432,0.3165,0.364112,0.209443,0.080317,0.029629,1.090433,-0.442580
80,Lisandro Martínez,Argentina,DEF,0.936360,0.3165,1.150432,0.3165,0.364112,0.209443,0.080317,0.029629,1.077218,-0.437216
41,Dayot Upamecano,France,DEF,0.904273,0.3112,1.167319,0.3112,0.363270,0.212026,0.082501,0.031004,1.055576,-0.432712
50,Mike Maignan,France,GK,0.885303,0.3112,1.167319,0.3112,0.363270,0.212026,0.082501,0.031004,1.033431,-0.423635
0,Cristian Romero,Argentina,DEF,0.898839,0.3165,1.150432,0.3165,0.364112,0.209443,0.080317,0.029629,1.034053,-0.419697


In [229]:
raw_save = (
    0.35 * df["expected_goals_conceded_norm"]  # opportunity
  + 0.40 * df["saves_per90"]             # ability
  + 0.05 * df["recent_saves_per90"]      # recent form
  + 0.20 * df["price"]                   # overall GK quality
)

# Expected saves (λ)
df["expected_saves"] = (
    sigmoid((raw_save - 0.5) * 6) * 4
) * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "expected_saves"] = 0.0

In [230]:
cols = [
    "name",
    "team",
    "expected_saves",
    "expected_goals_conceded",
    "saves_per90",
    "recent_saves_per90",
    "price",
    "expected_goals_conceded_norm",
]

df[df["position"] == "GK"][cols] \
    .sort_values("expected_saves", ascending=False) \
    .head(30)

,name,team,expected_saves,expected_goals_conceded,saves_per90,recent_saves_per90,price,expected_goals_conceded_norm
74,Unai Simón,Spain,3.088678,1.314532,0.615741,0.476190,1.000000,1.000000
50,Mike Maignan,France,2.785762,1.167319,0.791667,0.634921,1.000000,0.483089
9,Emiliano Martínez,Argentina,2.681947,1.150432,0.633333,0.909091,1.000000,0.423791
32,Jordan Pickford,England,2.466074,1.029739,1.000000,1.000000,0.866667,0.000000
73,David Raya,Spain,0.309517,1.314532,0.000000,0.000000,1.000000,1.000000
75,Joan García,Spain,0.184827,1.314532,0.000000,0.000000,0.333333,1.000000
51,Brice Samba,France,0.114586,1.167319,0.000000,0.000000,0.666667,0.483089
10,Gerónimo Rulli,Argentina,0.109147,1.150432,0.000000,0.000000,0.666667,0.423791
11,Juan Musso,Argentina,0.096026,1.150432,0.000000,0.000000,0.533333,0.423791
84,Robin Risser,France,0.065052,1.167319,0.000000,0.000000,0.000000,0.483089


In [231]:
raw_pen_save = (
    0.55 * df["price"]                    
  + 0.30 * df["expected_goals_conceded_norm"]
  + 0.15 * df["stat_PS"]
)

# Strong compression because penalty saves are exceptionally rare
df["prob_penalty_save"] = sigmoid((raw_pen_save - 0.5) * 3) / 20 * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "prob_penalty_save"] = 0.0

In [232]:
cols = [
    "name",
    "team",
    "prob_penalty_save",
    "price",
    "expected_goals_conceded",
    "opp_over_05_prob",
    "prob_plays_60min",
    "stat_PS",
]

df[df["position"] == "GK"][cols] \
    .sort_values("prob_penalty_save", ascending=False) \
    .head(30)

,name,team,prob_penalty_save,price,expected_goals_conceded,opp_over_05_prob,prob_plays_60min,stat_PS
74,Unai Simón,Spain,0.032791,1.000000,1.314532,1.000000,0.885303,0.0
50,Mike Maignan,France,0.032661,1.000000,1.167319,0.497081,0.885303,1.0
9,Emiliano Martínez,Argentina,0.029848,1.000000,1.150432,0.438503,0.947846,0.0
32,Jordan Pickford,England,0.021970,0.866667,1.029739,0.000000,0.910668,0.0
73,David Raya,Spain,0.004989,1.000000,1.314532,1.000000,0.134703,0.0
10,Gerónimo Rulli,Argentina,0.003181,0.666667,1.150432,0.438503,0.128420,0.0
51,Brice Samba,France,0.003113,0.666667,1.167319,0.497081,0.122389,0.0
75,Joan García,Spain,0.002983,0.333333,1.314532,1.000000,0.122389,0.0
11,Juan Musso,Argentina,0.002829,0.533333,1.150432,0.438503,0.128420,0.0
33,Dean Henderson,England,0.001990,0.466667,1.029739,0.000000,0.122389,0.0


In [233]:
raw_tackles = (
    0.50 * df["tackles_per90"]
  + 0.15 * df["recent_tackles_per90"]
  + 0.15 * df["expected_goals_conceded_norm"]
  + 0.20 * df["match_over_25_prob"]
)

df["lambda_tackles"] = -np.log(
    (1 - raw_tackles).clip(lower=0.001)
)

df["expected_tackles"] = (
    df["lambda_tackles"].clip(lower=0)
    * df["prob_plays_60min"] * 3
)

df.loc[df["position"] != "MID", "expected_tackles"] = 0.0

In [234]:
cols = [
    "name",
    "team",
    "expected_tackles",
    "tackles_per90",
    "recent_tackles_per90",
    "expected_goals_conceded",
    "match_over_25_prob",
    "price",
    "prob_plays_60min",
]

df[df["position"] == "MID"][cols] \
    .sort_values("expected_tackles", ascending=False) \
    .head(30)

,name,team,expected_tackles,tackles_per90,recent_tackles_per90,expected_goals_conceded,match_over_25_prob,price,prob_plays_60min
77,Rodri,Spain,1.398812,0.212291,0.333333,1.314532,0.4958,0.489796,0.897180
68,Lamine Yamal,Spain,1.054628,0.118519,0.203774,1.314532,0.4958,1.000000,0.849180
79,Pedri,Spain,1.001206,0.125874,0.172249,1.314532,0.4958,0.612245,0.809270
18,Alexis Mac Allister,Argentina,0.961134,0.200371,0.272727,1.150432,0.4215,0.306122,0.939439
78,Dani Olmo,Spain,0.924064,0.105263,0.185950,1.314532,0.4958,0.530612,0.770041
61,Álex Baena,Spain,0.883054,0.128049,0.223881,1.314532,0.4958,0.183673,0.691256
59,Michael Olise,France,0.721173,0.086066,0.135849,1.167319,0.4958,0.897959,0.897238
36,Elliot Anderson,England,0.663135,0.180113,0.284211,1.029739,0.4215,0.285714,0.903674
19,Enzo Fernández,Argentina,0.607066,0.087500,0.060000,1.150432,0.4215,0.489796,0.903705
55,Manu Koné,France,0.573273,0.140762,0.167702,1.167319,0.4958,0.204082,0.614781


In [235]:
raw_cc = (
    0.25 * df["recent_cc_per90"]
  + 0.25 * df["stat_CC"]
  + 0.25 * df["match_over_25_prob"]
  + 0.15 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_cc"] = -np.log(
    (1 - raw_cc).clip(lower=0.001)
)

df["expected_chances_created"] = (
    df["lambda_cc"].clip(lower=0)
    * df["prob_plays_60min"] * 2
)

df.loc[df["position"] != "MID", "expected_chances_created"] = 0.0

In [236]:
cols = [
    "name",
    "team",
    "position",
    "expected_chances_created",
    "cc_per90",
    "recent_cc_per90",
    "stat_CC",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_chances_created", ascending=False).head(30)

,name,team,position,expected_chances_created,cc_per90,recent_cc_per90,stat_CC,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
59,Michael Olise,France,MID,1.262585,0.625000,0.306918,0.714286,0.4958,0.2410,0.897959,0.897238
47,Ousmane Dembélé,France,MID,1.061144,0.396104,0.326640,0.428571,0.4958,0.3125,1.000000,0.862021
38,Jude Bellingham,England,MID,0.777225,0.355340,0.139748,0.428571,0.4215,0.2410,0.653061,0.905626
37,Declan Rice,England,MID,0.700711,0.474093,0.363095,0.428571,0.4215,0.1176,0.387755,0.785850
27,Bukayo Saka,England,MID,0.691580,0.685393,0.505176,0.428571,0.4215,0.1852,0.897959,0.567316
78,Dani Olmo,Spain,MID,0.601719,0.356725,0.168044,0.285714,0.4958,0.2198,0.530612,0.770041
29,Anthony Gordon,England,MID,0.597446,0.373089,0.428070,0.285714,0.4215,0.1818,0.387755,0.693773
68,Lamine Yamal,Spain,MID,0.534736,0.000000,0.000000,0.000000,0.4958,0.3077,1.000000,0.849180
79,Pedri,Spain,MID,0.441057,0.142191,0.000000,0.142857,0.4958,0.1176,0.612245,0.809270
49,Désiré Doué,France,MID,0.428199,0.190625,0.336088,0.142857,0.4958,0.2410,0.489796,0.536978


In [237]:
raw_sot = (
    0.35 * df["sot_per90"]
  + 0.15 * df["stat_GS"]
  + 0.20 * df["match_over_25_prob"]
  + 0.20 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_sot"] = -np.log(
    (1 - raw_sot).clip(lower=0.001)
)

df["expected_shots_on_target"] = (
    df["lambda_sot"].clip(lower=0)
    * df["prob_plays_60min"] * 2
)

df.loc[df["position"] != "FWD", "expected_shots_on_target"] = 0.0

In [238]:
cols = [
    "name",
    "team",
    "position",
    "expected_shots_on_target",
    "sot_per90",
    "goal_rate",
    "stat_GS",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_shots_on_target", ascending=False).head(30)

,name,team,position,expected_shots_on_target,sot_per90,goal_rate,stat_GS,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
46,Kylian Mbappé,France,FWD,2.283289,0.770270,0.926641,1.000,0.4958,0.5051,1.000000,0.897406
4,Lionel Messi,Argentina,FWD,2.003450,0.713208,0.905660,1.000,0.4215,0.3968,0.923077,0.939772
26,Harry Kane,England,FWD,1.412013,0.447602,0.639432,0.750,0.4215,0.3922,1.000000,0.930088
70,Mikel Oyarzabal,Spain,FWD,1.129556,0.447761,0.511727,0.500,0.4958,0.3774,0.630769,0.891084
5,Julián Alvarez,Argentina,FWD,0.581614,0.205379,0.146699,0.125,0.4215,0.2500,0.707692,0.829565
83,Lautaro Martínez,Argentina,FWD,0.359351,0.208609,0.397351,0.250,0.4215,0.2564,0.738462,0.465996
88,Jean-Philippe Mateta,France,FWD,0.355629,1.000000,0.000000,0.000,0.4958,0.3077,0.384615,0.223204
69,Ferran Torres,Spain,FWD,0.292865,0.200957,0.000000,0.000,0.4958,0.2817,0.584615,0.437779
31,Noni Madueke,England,FWD,0.198769,0.072917,0.000000,0.000,0.4215,0.1538,0.323077,0.523585
91,Ollie Watkins,England,FWD,0.086062,0.000000,0.000000,0.000,0.4215,0.3546,0.600000,0.177555


In [239]:
df["expected_qualification_points"] = df["team_qualify_prob"] * 0

In [240]:
cols = [
    "name",
    "team",
    "position",
    "expected_qualification_points",
    "team_qualify_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("team_qualify_prob", ascending=False).groupby("team").head(3).head(30)

,name,team,position,expected_qualification_points,team_qualify_prob,price,prob_plays_60min
40,Maxence Lacroix,France,DEF,0.0,0.625000,0.333333,0.157095
41,Dayot Upamecano,France,DEF,0.0,0.625000,0.866667,0.904273
42,Ibrahima Konaté,France,DEF,0.0,0.625000,0.600000,0.113515
31,Noni Madueke,England,FWD,0.0,0.588235,0.323077,0.523585
30,Morgan Rogers,England,MID,0.0,0.588235,0.428571,0.210939
27,Bukayo Saka,England,MID,0.0,0.588235,0.897959,0.567316
6,Giuliano Simeone,Argentina,MID,0.0,0.531915,0.102041,0.144045
7,Nico González,Argentina,MID,0.0,0.531915,0.102041,0.417776
0,Cristian Romero,Argentina,DEF,0.0,0.531915,0.600000,0.898839
61,Álex Baena,Spain,MID,0.0,0.495050,0.183673,0.691256


In [241]:
df.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {df.shape[0]} rows x {df.shape[1]} cols")


Saved fantasy_enriched.csv 100 rows x 265 cols


In [242]:
BUDGET = 105.0
MAX_PER_COUNTRY = 6
SQUAD_SIZE = 15
XI_SIZE = 11
BENCH_WEIGHT = 0.9

def compute_total_expected_points(df):
    df = df.copy()

    cs_value = {"GK": 5, "DEF": 5, "MID": 1, "FWD": 0}
    goal_value = {"GK": 9, "DEF": 7, "MID": 6, "FWD": 5}

    df["goal_pts_value"] = df["position"].map(goal_value)
    df["cs_pts_value"] = df["position"].map(cs_value)

    # appearance: 1pt for playing any, +1 if 60+
    e_appearance = df["prob_plays_any"] * 1 + df["prob_plays_60min"] * 1

    # goal
    e_goal = df["prob_scores"] * df["goal_pts_value"]

    # assist
    e_assist = df["expected_assists"] * 3

    # clean sheet: gated by 60min, value depends on position
    e_cs = df["prob_clean_sheet"] * df["cs_pts_value"]

    # goals conceded penalty: GK and DEF only
    e_gc = df["expected_gc_penalty"].copy()
    e_gc[~df["position"].isin(["GK", "DEF"])] = 0.0

    # saves bonus: GK only, every 3 saves = +1
    e_saves = (df["expected_saves"] / 3).copy()
    e_saves[df["position"] != "GK"] = 0.0

    # penalty save: GK only
    e_pen_save = (df["prob_penalty_save"] * 3).copy()
    e_pen_save[df["position"] != "GK"] = 0.0

    # tackles bonus: MID only, every 3 tackles = +1
    e_tackles = (df["expected_tackles"] / 3).copy()
    e_tackles[df["position"] != "MID"] = 0.0

    # big chances created: MID only, every 2 = +1
    e_cc = (df["expected_chances_created"] / 2).copy()
    e_cc[df["position"] != "MID"] = 0.0

    # shots on target: FWD only, every 2 = +1
    e_sot = (df["expected_shots_on_target"] / 2).copy()
    e_sot[df["position"] != "FWD"] = 0.0

    # yellow card
    e_yc = df["prob_yellow_card"] * -1

    # red card
    e_rc = df["prob_red_card"] * -2

    # own goal
    e_og = df["prob_own_goal"] * -2

    # penalty won
    e_pw = df["prob_pen_won"] * 2

    # penalty committed
    e_pc = df["prob_penalty_committed"] * -1

    # qualification bonus: +2 per player in XI who advances, requires playing
    # captain's quali bonus is NOT doubled per rules
    e_quali = df["expected_qualification_points"] * df["prob_plays_any"]

    # scouting bonus: +2 if differential==1 AND player scores >4 base pts
    # model: use Poisson confidence interval on expected base points
    # base pts = everything except scouting and captain multiplier
    base_ep = (
        e_appearance + e_goal + e_assist + e_cs + e_gc +
        e_saves + e_pen_save + e_tackles + e_cc + e_sot +
        e_yc + e_rc + e_og + e_pw + e_pc + e_quali
    )

    # Poisson 90% CI: lower = ppf(0.05, mu), upper = ppf(0.95, mu)
    mu = base_ep.clip(lower=0.01)
    lower_ci = pd.Series(poisson.ppf(0.05, mu), index=df.index)
    upper_ci = pd.Series(poisson.ppf(0.95, mu), index=df.index)

    # if lower CI > 4: full +2 expected
    # if upper CI < 4: 0
    # if CI straddles 4: scale by P(X>4 | mu) * 2
    p_exceeds_4 = pd.Series(1 - poisson.cdf(4, mu), index=df.index)

    scouting_ep = pd.Series(0.0, index=df.index)
    eligible = df["differential"] == 1
    full_bonus = eligible & (lower_ci > 4)
    no_bonus = eligible & (upper_ci < 4)
    partial = eligible & ~full_bonus & ~no_bonus

    scouting_ep[full_bonus] = 2.0
    scouting_ep[no_bonus] = 0.0
    scouting_ep[partial] = p_exceeds_4[partial] * 2.0

    df["e_appearance"] = e_appearance
    df["e_goal"] = e_goal
    df["e_assist"] = e_assist
    df["e_cs"] = e_cs
    df["e_gc"] = e_gc
    df["e_saves"] = e_saves
    df["e_pen_save"] = e_pen_save
    df["e_tackles"] = e_tackles
    df["e_cc"] = e_cc
    df["e_sot"] = e_sot
    df["e_yc"] = e_yc
    df["e_rc"] = e_rc
    df["e_og"] = e_og
    df["e_pw"] = e_pw
    df["e_pc"] = e_pc
    df["e_quali"] = e_quali
    df["e_scouting"] = scouting_ep

    e_elim_risk = -1 * (1 - df["team_qualify_prob"])
    df["e_elim_risk"] = e_elim_risk

    df["expected_points"] = base_ep + scouting_ep + e_elim_risk

    return df

In [243]:

def optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):
    idx = df.index.tolist()

    x = LpVariable.dicts("squad", idx, cat="Binary")
    s = LpVariable.dicts("xi", idx, cat="Binary")
    b = LpVariable.dicts("bench", idx, cat="Binary")    
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("fantasy", LpMaximize)

    
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
        for i in idx
    )

    # squad = 15, xi = 11, bench = 4
    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    # budget
    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    # squad composition: 2 GK, 5 DEF, 5 MID, 3 FWD
    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    # country limit
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country


    # max 2 GK+DEF combined from the same country
    for country in df["team"].unique():
        gkdef_idx = df[(df["team"] == country) & (df["position"].isin(["GK", "DEF"]))].index.tolist()
        model += lpSum(x[i] for i in gkdef_idx) <= 2

    # XI formation: 1 GK starts, valid outfield formation
    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    # captain: exactly 1, must be in XI
    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad = [i for i in idx if value(x[i]) > 0.5]
    in_xi = [i for i in idx if value(s[i]) > 0.5]
    on_bench = [i for i in idx if value(b[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "squad": df.loc[in_squad].copy(),
        "xi": df.loc[in_xi].copy(),
        "bench": df.loc[on_bench].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
        "cost": df.loc[in_squad, "price_raw"].sum(),
    }

In [244]:
def optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY):
    # best possible 11 ignoring bench constraint, just 11 players
    idx = df.index.tolist()

    s = LpVariable.dicts("xi", idx, cat="Binary")
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("ideal_xi", LpMaximize)

    # optimize_ideal_xi objective — NO bench term
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        for i in idx
    )

    model += lpSum(s[i] for i in idx) == 11

    # no budget constraint for ideal XI comparison
    # country limit still applies
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(s[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        return None

    in_xi = [i for i in idx if value(s[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "xi": df.loc[in_xi].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
    }


def print_team(result, ideal=None):
    pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
    cap = result["captain"]

    xi = result["xi"].copy()
    xi["_ord"] = xi["position"].map(pos_order)
    xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])

    bench = result["bench"].copy()
    bench["_ord"] = bench["position"].map(pos_order)
    bench = bench.sort_values(["_ord", "expected_points"], ascending=[True, False])

    print(f"\nExpected points: {result['total_ep']:.2f}   Cost: ${result['cost']:.1f}M\n")

    print("Starting XI")
    for i, row in xi.iterrows():
        tag = " [C]" if i == cap else ""
        scout = " [SCOUT]" if row.get("differential", 0) == 1 else ""
        print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{scout}")

    print("\nBench")
    for rank, (i, row) in enumerate(bench.iterrows(), 1):
        print(f"  [{rank}] {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}")

    print(f"\nCountry breakdown: {dict(result['squad']['team'].value_counts())}")

    if ideal:
        ideal_xi = ideal["xi"].copy()
        ideal_xi["_ord"] = ideal_xi["position"].map(pos_order)
        ideal_xi = ideal_xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        ideal_cap = ideal["captain"]

        print(f"\nIdeal XI (no budget constraint, 11 only, EP: {ideal['total_ep']:.2f})")
        for i, row in ideal_xi.iterrows():
            tag = " [C]" if i == ideal_cap else ""
            in_squad = i in result["squad"].index
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")
            
if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)

    df = compute_total_expected_points(df)

    result = optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)

    print_team(result, ideal)


Expected points: 83.22   Cost: $105.0M

Starting XI
  GK   Mike Maignan                 France                 $5.0  EP:3.36
  DEF  Marc Cucurella               Spain                  $5.1  EP:3.66
  DEF  Jules Koundé                 France                 $5.4  EP:3.37
  DEF  Lisandro Martínez            Argentina              $4.6  EP:2.99
  MID  Michael Olise                France                 $9.5  EP:6.90
  MID  Ousmane Dembélé              France                 $10.0  EP:6.80
  MID  Jude Bellingham              England                $8.3  EP:6.71
  MID  Dani Olmo                    Spain                  $7.7  EP:4.94 [SCOUT]
  FWD  Lionel Messi                 Argentina              $10.0  EP:9.27 [C]
  FWD  Kylian Mbappé                France                 $10.5  EP:8.99
  FWD  Mikel Oyarzabal              Spain                  $8.1  EP:5.54

Bench
  [1] GK   Jordan Pickford              England                $4.8  EP:3.01
  [2] DEF  Cristian Romero              Argen

In [245]:
def optimize_with_transfers(df, current_team_names, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):

    current_idx = set()
    unmatched = []
    for name in current_team_names:
        match = df[df["name"].str.lower().str.strip() == name.lower().strip()]
        if len(match) == 0:
            match = df[df["name"].str.lower().str.contains(name.lower().strip(), na=False)]
        if len(match) > 0:
            current_idx.add(match.index[0])
        else:
            unmatched.append(name)

  
    if unmatched:
        print(f"Eliminated players: {unmatched}")

    forced_transfers = len(unmatched)

    idx = df.index.tolist()

    x         = LpVariable.dicts("squad", idx, cat="Binary")
    s         = LpVariable.dicts("xi",    idx, cat="Binary")
    b         = LpVariable.dicts("bench", idx, cat="Binary")
    cap       = LpVariable.dicts("cap",   idx, cat="Binary")
    transfer_out = LpVariable.dicts("out", idx, cat="Binary")

    model = LpProblem("fantasy_transfers", LpMaximize)

    transfers_made = lpSum(transfer_out[i] for i in idx)   # voluntary drops only

    n_extra = LpVariable("n_extra", lowBound=0)

    # CHANGE 2: total transfers = voluntary + forced. Penalty fires when
    # (voluntary + forced) > free_transfers, not just voluntary > free_transfers.
    model += n_extra >= transfers_made + forced_transfers - free_transfers

    model += (
        lpSum(
            df.loc[i, "expected_points"] * s[i]
            + df.loc[i, "expected_points"] * cap[i]
            + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
            for i in idx
        )
        - 3 * n_extra
    )

    for i in idx:
        if i in current_idx:
            model += transfer_out[i] == 1 - x[i]
        else:
            model += transfer_out[i] == 0

    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad  = [i for i in idx if value(x[i]) > 0.5]
    in_xi     = [i for i in idx if value(s[i]) > 0.5]
    on_bench  = [i for i in idx if value(b[i]) > 0.5]
    captain   = next(i for i in idx if value(cap[i]) > 0.5)

    new_squad_idx  = set(in_squad)
    voluntary_out  = [i for i in current_idx if i not in new_squad_idx]
    transfers_in   = [i for i in new_squad_idx if i not in current_idx]

    # CHANGE 3: total = voluntary drops of matched players + forced drops of eliminated.
    n_transfers = len(voluntary_out) + forced_transfers
    penalty     = max(0, n_transfers - free_transfers) * 3

    return {
        "squad":         df.loc[in_squad].copy(),
        "xi":            df.loc[in_xi].copy(),
        "bench":         df.loc[on_bench].copy(),
        "captain":       captain,
        "total_ep":      value(model.objective),
        "cost":          df.loc[in_squad, "price_raw"].sum(),
        "transfers_out": df.loc[voluntary_out, "name"].tolist() + unmatched,  # eliminated shown in OUT
        "transfers_in":  df.loc[transfers_in, "name"].tolist(),
        "n_transfers":   n_transfers,
        "penalty":       penalty,
        "forced_out":    unmatched,   # separate field for clarity
    }

def print_transfers(result):
    if result is None:
        print("No solution.")
        return

    print(f"\nTransfers made: {result['n_transfers']} ({result['n_transfers'] - 5} penalised at -3 each)" if result['n_transfers'] > 5 else f"\nTransfers made: {result['n_transfers']} (all free)")
    print(f"Point penalty: -{result['penalty']}")

    if result["transfers_out"]:
        print("\nOUT:")
        for name in result["transfers_out"]:
            print(f"  {name}")
        print("IN:")
        for name in result["transfers_in"]:
            print(f"  {name}")

    print_team(result)


if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)
    df = compute_total_expected_points(df)

    current_team = [
        "Emiliano Martínez", "Lisandro Martínez", "Achraf Hakimi", "Nico O'Reilly", "Marc Cucurella",
        "Ousmane Dembélé", "Michael Olise", "Unai Simón", "Jude Bellingham", "Kylian Mbappé", "Mikel Oyarzabal",
        "Dani Olmo", "Facundo Medina", "Lionel Messi", "Brahim Díaz"
    ]

    result = optimize_with_transfers(df, current_team, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)
    print_transfers(result)

    print("\nIdeal XI for comparison:")
    if ideal:
        pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
        xi = ideal["xi"].copy()
        xi["_ord"] = xi["position"].map(pos_order)
        xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        cap = ideal["captain"]
        for i, row in xi.iterrows():
            tag = " [C]" if i == cap else ""
            in_squad = i in result["squad"].index if result else False
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")

Eliminated players: ['Achraf Hakimi', 'Facundo Medina', 'Brahim Díaz']

Transfers made: 5 (all free)
Point penalty: -0

OUT:
  Emiliano Martínez
  Unai Simón
  Achraf Hakimi
  Facundo Medina
  Brahim Díaz
IN:
  Cristian Romero
  Jordan Pickford
  Mike Maignan
  Adrien Rabiot
  Jules Koundé

Expected points: 83.22   Cost: $105.0M

Starting XI
  GK   Mike Maignan                 France                 $5.0  EP:3.36
  DEF  Marc Cucurella               Spain                  $5.1  EP:3.66
  DEF  Jules Koundé                 France                 $5.4  EP:3.37
  DEF  Lisandro Martínez            Argentina              $4.6  EP:2.99
  MID  Michael Olise                France                 $9.5  EP:6.90
  MID  Ousmane Dembélé              France                 $10.0  EP:6.80
  MID  Jude Bellingham              England                $8.3  EP:6.71
  MID  Dani Olmo                    Spain                  $7.7  EP:4.94 [SCOUT]
  FWD  Lionel Messi                 Argentina              $10.

In [246]:
def optimize_with_transfers(df, current_team_names, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):

    current_idx = set()
    unmatched = []
    for name in current_team_names:
        match = df[df["name"].str.lower().str.strip() == name.lower().strip()]
        if len(match) == 0:
            match = df[df["name"].str.lower().str.contains(name.lower().strip(), na=False)]
        if len(match) > 0:
            current_idx.add(match.index[0])
        else:
            unmatched.append(name)

  
    if unmatched:
        print(f"Eliminated players: {unmatched}")

    forced_transfers = len(unmatched)

    idx = df.index.tolist()

    x         = LpVariable.dicts("squad", idx, cat="Binary")
    s         = LpVariable.dicts("xi",    idx, cat="Binary")
    b         = LpVariable.dicts("bench", idx, cat="Binary")
    cap       = LpVariable.dicts("cap",   idx, cat="Binary")
    transfer_out = LpVariable.dicts("out", idx, cat="Binary")

    model = LpProblem("fantasy_transfers", LpMaximize)

    transfers_made = lpSum(transfer_out[i] for i in idx)   # voluntary drops only

    n_extra = LpVariable("n_extra", lowBound=0)

    # CHANGE 2: total transfers = voluntary + forced. Penalty fires when
    # (voluntary + forced) > free_transfers, not just voluntary > free_transfers.
    model += n_extra >= transfers_made + forced_transfers - free_transfers

    model += (
        lpSum(
            df.loc[i, "expected_points"] * s[i]
            + df.loc[i, "expected_points"] * cap[i]
            + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
            for i in idx
        )
        - 3 * n_extra
    )

    for i in idx:
        if i in current_idx:
            model += transfer_out[i] == 1 - x[i]
        else:
            model += transfer_out[i] == 0

    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad  = [i for i in idx if value(x[i]) > 0.5]
    in_xi     = [i for i in idx if value(s[i]) > 0.5]
    on_bench  = [i for i in idx if value(b[i]) > 0.5]
    captain   = next(i for i in idx if value(cap[i]) > 0.5)

    new_squad_idx  = set(in_squad)
    voluntary_out  = [i for i in current_idx if i not in new_squad_idx]
    transfers_in   = [i for i in new_squad_idx if i not in current_idx]

    # CHANGE 3: total = voluntary drops of matched players + forced drops of eliminated.
    n_transfers = len(voluntary_out) + forced_transfers
    penalty     = max(0, n_transfers - free_transfers) * 3

    return {
        "squad":         df.loc[in_squad].copy(),
        "xi":            df.loc[in_xi].copy(),
        "bench":         df.loc[on_bench].copy(),
        "captain":       captain,
        "total_ep":      value(model.objective),
        "cost":          df.loc[in_squad, "price_raw"].sum(),
        "transfers_out": df.loc[voluntary_out, "name"].tolist() + unmatched,  # eliminated shown in OUT
        "transfers_in":  df.loc[transfers_in, "name"].tolist(),
        "n_transfers":   n_transfers,
        "penalty":       penalty,
        "forced_out":    unmatched,   # separate field for clarity
    }

def print_transfers(result):
    if result is None:
        print("No solution.")
        return

    print(f"\nTransfers made: {result['n_transfers']} ({result['n_transfers'] - 5} penalised at -3 each)" if result['n_transfers'] > 5 else f"\nTransfers made: {result['n_transfers']} (all free)")
    print(f"Point penalty: -{result['penalty']}")

    if result["transfers_out"]:
        print("\nOUT:")
        for name in result["transfers_out"]:
            print(f"  {name}")
        print("IN:")
        for name in result["transfers_in"]:
            print(f"  {name}")

    print_team(result)


if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)
    df = compute_total_expected_points(df)

    current_team = [
        "Mike Maignan", "Lisandro Martínez", "Dayot Upamecano", "Nico O'Reilly", "Marc Cucurella",
        "Ousmane Dembélé", "Michael Olise", "Jude Bellingham", "Harry Kane", "Kylian Mbappé", "Álex Baena",
        "Unai Simón", "Cristian Romero", "Lionel Messi", "Patrick Berg"
    ]

    result = optimize_with_transfers(df, current_team, free_transfers=5, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)
    print_transfers(result)

Eliminated players: ['Patrick Berg']

Transfers made: 5 (all free)
Point penalty: -0

OUT:
  Dayot Upamecano
  Unai Simón
  Harry Kane
  Álex Baena
  Patrick Berg
IN:
  Jordan Pickford
  Mikel Oyarzabal
  Dani Olmo
  Adrien Rabiot
  Jules Koundé

Expected points: 83.22   Cost: $105.0M

Starting XI
  GK   Mike Maignan                 France                 $5.0  EP:3.36
  DEF  Marc Cucurella               Spain                  $5.1  EP:3.66
  DEF  Jules Koundé                 France                 $5.4  EP:3.37
  DEF  Lisandro Martínez            Argentina              $4.6  EP:2.99
  MID  Michael Olise                France                 $9.5  EP:6.90
  MID  Ousmane Dembélé              France                 $10.0  EP:6.80
  MID  Jude Bellingham              England                $8.3  EP:6.71
  MID  Dani Olmo                    Spain                  $7.7  EP:4.94 [SCOUT]
  FWD  Lionel Messi                 Argentina              $10.0  EP:9.27 [C]
  FWD  Kylian Mbappé          

In [247]:
for pos in ["GK", "DEF", "MID", "FWD"]:
    print(f"\n-{pos}")
    print(df[df["position"] == pos].nlargest(10, "expected_points")[["name", "team", "price_raw", "expected_points"]].to_string(index=False))


-GK
             name      team  price_raw  expected_points
     Mike Maignan    France        5.0         3.356498
       Unai Simón     Spain        5.0         3.071866
  Jordan Pickford   England        4.8         3.007354
Emiliano Martínez Argentina        5.0         2.840873
   James Trafford   England        4.0         0.192098
     Robin Risser    France        3.5         0.173132
      Brice Samba    France        4.5         0.165643
   Dean Henderson   England        4.2         0.157291
       David Raya     Spain        5.0         0.129874
   Gerónimo Rulli Argentina        4.5         0.109481

-DEF
             name      team  price_raw  expected_points
   Marc Cucurella     Spain        5.1         3.661446
     Jules Koundé    France        5.4         3.365207
  Dayot Upamecano    France        5.3         3.359352
   William Saliba    France        5.3         3.075589
       Marc Guéhi   England        5.1         3.034629
Lisandro Martínez Argentina        4.

In [248]:
player1 = "Mike Maignan"
player2 = "Unai Simón"

cols = [
    "name", "team", "position", "price_raw", "expected_points",
    "e_appearance", "e_goal", "e_assist", "e_cs", "e_gc",
    "e_saves", "e_pen_save", "e_tackles", "e_cc", "e_sot",
    "e_yc", "e_rc", "e_og", "e_pw", "e_pc",
    "e_quali", "e_scouting"
]

comparison = df.loc[
    df["name"].isin([player1, player2]),
    cols
].set_index("name").T

print(comparison)

name            Mike Maignan Unai Simón
team                  France      Spain
position                  GK         GK
price_raw                5.0        5.0
expected_points     3.356498   3.071866
e_appearance        1.739421   1.726515
e_goal                   0.0        0.0
e_assist            0.020491   0.016781
e_cs                1.462654    1.45827
e_gc               -0.423635  -0.516249
e_saves             0.928587   1.029559
e_pen_save          0.097982   0.098372
e_tackles                0.0        0.0
e_cc                     0.0        0.0
e_sot                    0.0        0.0
e_yc               -0.101293   -0.19034
e_rc               -0.026617  -0.047895
e_og               -0.016197  -0.033272
e_pw                 0.06784   0.062699
e_pc               -0.017735  -0.027624
e_quali                  0.0        0.0
e_scouting               0.0        0.0
